In [37]:
import sys
import gymnasium as gym
import logging
import numpy as np
np.random.seed(0)

root_logger = logging.getLogger()
if root_logger.handlers:
    for handler in root_logger.handlers:
        root_logger.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    stream=sys.stdout,  # 显式指定输出流
    datefmt='%H:%M:%S',
    force=True  # 强制覆盖已有的日志配置（关键参数）
)

env = gym.make('MountainCarContinuous-v0')
for key in vars(env):
    logging.info('%s: %s', key, vars(env)[key])
for key in vars(env.spec):
    logging.info('%s: %s', key, vars(env.spec)[key])

21:07:08 [INFO] _saved_kwargs: {'max_episode_steps': 999}
21:07:08 [INFO] env: <OrderEnforcing<PassiveEnvChecker<Continuous_MountainCarEnv<MountainCarContinuous-v0>>>>
21:07:08 [INFO] _action_space: None
21:07:08 [INFO] _observation_space: None
21:07:08 [INFO] _metadata: None
21:07:08 [INFO] _cached_spec: None
21:07:08 [INFO] _max_episode_steps: 999
21:07:08 [INFO] _elapsed_steps: None
21:07:08 [INFO] id: MountainCarContinuous-v0
21:07:08 [INFO] entry_point: gymnasium.envs.classic_control.continuous_mountain_car:Continuous_MountainCarEnv
21:07:08 [INFO] reward_threshold: 90.0
21:07:08 [INFO] nondeterministic: False
21:07:08 [INFO] max_episode_steps: 999
21:07:08 [INFO] order_enforce: True
21:07:08 [INFO] disable_env_checker: False
21:07:08 [INFO] kwargs: {}
21:07:08 [INFO] additional_wrappers: ()
21:07:08 [INFO] vector_entry_point: None
21:07:08 [INFO] namespace: None
21:07:08 [INFO] name: MountainCarContinuous
21:07:08 [INFO] version: 0


In [38]:
class ClosedFormAgent:
    def __init__(self, _):
        pass

    def reset(self, mode=None):
        pass

    def step(self, observation, reward, terminated):
        position, velocity = observation
        if position > -4 * velocity or position < 13 * velocity - 0.6:
            force = 1.
        else:
            force = -1.
        action = np.array([force,])
        return action

    def close(self):
        pass


agent = ClosedFormAgent(env)

In [39]:
def play_episode(env, agent, seed=None, mode=None, render=False):
    observation, _ = env.reset(seed=seed)
    reward, terminated, truncated = 0., False, False
    agent.reset(mode=mode)
    episode_reward, elapsed_steps = 0., 0
    while True:
        action = agent.step(observation, reward, terminated)
        if render:
            env.render()
        if terminated or truncated:
            break
        observation, reward, terminated, truncated, _ = env.step(action)
        episode_reward += reward
        elapsed_steps += 1
    agent.close()
    return episode_reward, elapsed_steps


logging.info('==== test ====')
episode_rewards = []
for episode in range(100):
    episode_reward, elapsed_steps = play_episode(env, agent)
    episode_rewards.append(episode_reward)
    logging.info('test episode %d: reward = %.2f, steps = %d',
            episode, episode_reward, elapsed_steps)
logging.info('average episode reward = %.2f ± %.2f',
        np.mean(episode_rewards), np.std(episode_rewards))

21:07:08 [INFO] ==== test ====
21:07:08 [INFO] test episode 0: reward = 93.40, steps = 66
21:07:08 [INFO] test episode 1: reward = 93.30, steps = 67
21:07:08 [INFO] test episode 2: reward = 93.40, steps = 66
21:07:08 [INFO] test episode 3: reward = 93.40, steps = 66
21:07:08 [INFO] test episode 4: reward = 93.40, steps = 66
21:07:08 [INFO] test episode 5: reward = 93.30, steps = 67
21:07:08 [INFO] test episode 6: reward = 93.30, steps = 67
21:07:08 [INFO] test episode 7: reward = 93.40, steps = 66
21:07:08 [INFO] test episode 8: reward = 93.40, steps = 66
21:07:08 [INFO] test episode 9: reward = 93.30, steps = 67
21:07:08 [INFO] test episode 10: reward = 93.40, steps = 66
21:07:08 [INFO] test episode 11: reward = 93.40, steps = 66
21:07:08 [INFO] test episode 12: reward = 93.30, steps = 67
21:07:08 [INFO] test episode 13: reward = 93.40, steps = 66
21:07:08 [INFO] test episode 14: reward = 93.40, steps = 66
21:07:08 [INFO] test episode 15: reward = 93.40, steps = 66
21:07:08 [INFO] tes

In [40]:
env.close()